# 从Stage2嵌入生成蛋白质序列

本 Notebook 将演示如何使用您训练好的 `VESM` 模型来执行一个两阶段的生成任务：
1.  **第一步**: 输入一个 `stage2` 的全局嵌入向量，模型将解码并恢复出每个蛋白质对应的 `stage1` 嵌入向量。
2.  **第二步**: 将恢复出的 `stage1` 嵌入向量作为输入，模型将进一步解码生成对应的氨基酸序列。

---

## 1. 环境设置与导入

首先，我们需要导入所有必要的库和您项目中的自定义模块。请确保此 Notebook 与您的 `.py` 文件（如 `modules.py`, `trainUtils.py` 等）位于同一目录下，或者该目录已添加到 Python 路径中。

In [1]:
import json
import torch
import numpy as np
import sys
import os

# 假设您的代码文件在 '00evoModel' 文件夹中
# 如果此 notebook 不在该目录下，请取消下面的注释并修改路径
# sys.path.append('path/to/your/00evoModel')

# 导入您的自定义模块
import modules
import trainUtils

# 导入ESM3词汇表用于序列解码
try:
    from esm.utils.constants import esm3 as C
    VOCAB = C.SEQUENCE_VOCAB
except ImportError:
    print("无法导入 ESM 库，将使用一个预定义的词汇表。")
    # 如果您没有安装 'fair-esm' 库，这里提供一个备用词汇表
    VOCAB = ['<cls>', '<pad>', '<eos>', '<unk>', 'L', 'A', 'G', 'V', 'S', 'E', 'R', 'T', 'I', 'D', 'P', 'K', 'Q', 'N', 'F', 'Y', 'M', 'H', 'W', 'C', 'X', 'B', 'U', 'Z', 'O', '.', '-', '<null_1>', '<mask>']

print("库和模块导入成功！")

库和模块导入成功！


## 2. 加载配置和模型

在这一步，我们将加载 `config.json` 文件来获取模型参数，并使用 `trainUtils` 中的函数来构建模型结构并加载您训练好的权重。

**请务必修改下面的 `config_path` 和 `checkpoint_path` 为您的实际文件路径。**

In [2]:
# --- 用户需要修改的路径 ---
config_path = "/data2/zhoukaitao/01evoModel/checkpoints/250907_stage2/config.json"
# 注意：这里的 checkpoint 应该是您训练完成的 Stage 1+2 模型的 .ckpt 文件
# config.json 文件中的 checkpoint 路径也会被加载，但这里的路径优先级更高
checkpoint_path = "/data2/zhoukaitao/01evoModel/checkpoints/250907_stage2/epoch=15-validation_loss=19.6634.ckpt"
# --------------------------

# 加载配置文件
with open(config_path, 'r') as f:
    configs = json.load(f)

print("成功加载配置文件:", config_path)

# 设置设备 (GPU or CPU)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"将使用设备: {device}")

# 加载预训练基础模型 (例如 ESM3)
print("加载预训练基础模型...")
pretrain_model = trainUtils.loadPretrainModel(configs)

# 构建完整的 VESM 模型并加载训练好的权重
print("构建 VESM 模型并加载 checkpoint...")
model = trainUtils.buildModel(configs, pretrain_model, checkpoint_path)
model.to(device)
model.eval()  # 设置为评估模式

print("模型加载完成并已设置为评估模式！")

成功加载配置文件: /data2/zhoukaitao/01evoModel/checkpoints/250907_stage2/config.json
将使用设备: cuda:0
加载预训练基础模型...
load local model: /data2/zhoukaitao/01evoModel/checkpoints/250813_stage1_test/pretrain_stage1_250824.pth
构建 VESM 模型并加载 checkpoint...
model at stage: training stage 2


/home/zhoukaitao/anaconda3/envs/esm3/lib/python3.10/site-packages/torch/nn/init.py:452: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


incompatible parameters []
finish loading parameters
load model from checkpoint /data2/zhoukaitao/01evoModel/checkpoints/250907_stage2/epoch=15-validation_loss=19.6634.ckpt
模型加载完成并已设置为评估模式！


## 3. 定义生成函数

下面的代码块定义了核心的生成函数 `generate_sequence_from_stage2`。它封装了从 `stage2` 嵌入到 `stage1` 嵌入，再到最终序列的整个解码过程。

In [3]:
def generate_sequence_from_stage2(model, stage2_embedding, max_seq_len=2000):
    """
    从一个 stage2 嵌入向量生成所有蛋白质的序列。

    Args:
        model (modules.VESM): 训练好的 VESM 模型。
        stage2_embedding (torch.Tensor): 一个 stage2 嵌入向量，形状为 (channels,) 或 (1, channels)。
        max_seq_len (int): 生成序列的最大长度。

    Returns:
        dict: 一个字典，键是蛋白质名称，值是生成的氨基酸序列字符串。
        dict: 一个字典，包含恢复的 stage1 嵌入。
    """
    
    # --- 准备工作 ---
    model.eval() # 确保是评估模式
    device = next(model.parameters()).device
    config = model.config
    protein_names = config.prots
    num_proteins = len(protein_names)
    
    # 确保输入 embedding 的维度正确 (batch_size=1, channels)
    if stage2_embedding.dim() == 1:
        stage2_embedding = stage2_embedding.unsqueeze(0)
    stage2_embedding = stage2_embedding.to(device)
    
    reconstructed_sequences = {}
    reconstructed_s1_embeddings = {}
    
    with torch.no_grad():
        # --- 步骤 1: 从 Stage2 嵌入恢复 Stage1 嵌入 ---
        print("--- 步骤 1: 正在从 Stage2 嵌入恢复 Stage1 嵌入 ---")
        
        # 模拟 stage2_forward 中的解码过程
        # 输入: (1, channels) -> 输出: (1, num_proteins, channels)
        s2_embed_expanded = stage2_embedding.unsqueeze(1).repeat(1, num_proteins, 1)
        
        reconstructed_s1_tensor = s2_embed_expanded
        for block in model.stage_2_decoder_blocks:
            reconstructed_s1_tensor = block(reconstructed_s1_tensor)
        
        for i, prot_name in enumerate(protein_names):
            reconstructed_s1_embeddings[prot_name] = reconstructed_s1_tensor[:, i, :]
        
        print("Stage1 嵌入恢复完成！\n")
        
        # --- 步骤 2: 从恢复的 Stage1 嵌入生成序列 ---
        print("--- 步骤 2: 正在从 Stage1 嵌入生成蛋白质序列 ---")
        
        for prot_name, s1_embedding in reconstructed_s1_embeddings.items():
            # 模拟 stage1_forward 中的解码过程
            # 输入: (1, channels) -> 输出: (1, max_seq_len, channels)
            s1_embed_expanded = s1_embedding.unsqueeze(1).repeat(1, max_seq_len, 1)
            
            # 通过 stage1 的解码器 (reconstructor)
            decoder_output = model.stage_1_reconstructor(s1_embed_expanded)
            logits = decoder_output['aa_logits'] # 获取氨基酸预测的 logits
            
            # 从 logits 中获取最可能的 token ID
            predicted_token_ids = torch.argmax(logits, dim=-1).squeeze(0) # (max_seq_len)
            
            # 将 token ID 转换回氨基酸字符
            sequence_chars = []
            for token_id in predicted_token_ids.cpu().numpy():
                token_char = VOCAB[token_id]
                # 遇到结束符或填充符则停止
                if token_char in ['<eos>', '<pad>']:
                    break
                # 忽略特殊 token
                if token_char not in ['<cls>', '<unk>', '<null_1>', '<mask>']:
                    sequence_chars.append(token_char)
            
            reconstructed_sequences[prot_name] = "".join(sequence_chars)
            print(f"已生成蛋白质 '{prot_name}' 的序列。")
            
    return reconstructed_sequences, reconstructed_s1_embeddings

## 4. 执行生成并查看结果

现在，我们将创建一个随机的 `stage2` 嵌入向量作为输入，然后调用我们刚刚定义的生成函数，并打印出最终生成的序列。

In [6]:
# 获取 Stage 2 嵌入的维度
out_channels = configs['model']['params']['out_channels']

# 创建一个随机的 Stage 2 嵌入作为示例输入
# 在实际应用中，您可以从您的上游任务（如变体预测、适应性分析等）获得这个嵌入
random_stage2_embedding = torch.randn(out_channels)

print(f"创建了一个随机的 Stage 2 嵌入向量，维度为: {random_stage2_embedding.shape}")
print("\n调用生成函数...")
print("="*30)

# 执行生成
generated_sequences, restored_s1_embeds = generate_sequence_from_stage2(model, random_stage2_embedding)

print("="*30)
print("\n生成流程完成！\n")

# 打印恢复的 Stage 1 嵌入信息 (以 S 蛋白为例)
print("恢复的 Stage 1 嵌入 (以 S 蛋白为例):")
if 'S' in restored_s1_embeds:
    print(f"  - 形状: {restored_s1_embeds['S'].shape}")
    print(f"  - 部分值: {restored_s1_embeds['S'][0, :10].cpu().numpy()}")
else:
    print("未能恢复 S 蛋白的嵌入。")

print("\n-- 最终生成的蛋白质序列 --\n")
for prot_name, sequence in generated_sequences.items():
    print(f">{prot_name} (长度: {len(sequence)})\n{sequence}\n")


创建了一个随机的 Stage 2 嵌入向量，维度为: torch.Size([1152])

调用生成函数...
--- 步骤 1: 正在从 Stage2 嵌入恢复 Stage1 嵌入 ---
Stage1 嵌入恢复完成！

--- 步骤 2: 正在从 Stage1 嵌入生成蛋白质序列 ---
已生成蛋白质 'NSP1' 的序列。
已生成蛋白质 'NSP2' 的序列。
已生成蛋白质 'NSP3' 的序列。
已生成蛋白质 'NSP4' 的序列。
已生成蛋白质 'NSP5' 的序列。
已生成蛋白质 'NSP6' 的序列。
已生成蛋白质 'NSP7' 的序列。
已生成蛋白质 'NSP8' 的序列。
已生成蛋白质 'NSP9' 的序列。
已生成蛋白质 'NSP10' 的序列。
已生成蛋白质 'NSP11' 的序列。
已生成蛋白质 'NSP12' 的序列。
已生成蛋白质 'NSP13' 的序列。
已生成蛋白质 'NSP14' 的序列。
已生成蛋白质 'NSP15' 的序列。
已生成蛋白质 'NSP16' 的序列。
已生成蛋白质 'S' 的序列。
已生成蛋白质 'ORF3a' 的序列。
已生成蛋白质 'E' 的序列。
已生成蛋白质 'M' 的序列。
已生成蛋白质 'ORF6' 的序列。
已生成蛋白质 'ORF7a' 的序列。
已生成蛋白质 'ORF7b' 的序列。
已生成蛋白质 'ORF8' 的序列。
已生成蛋白质 'N' 的序列。
已生成蛋白质 'ORF9b' 的序列。
已生成蛋白质 'ORF10' 的序列。

生成流程完成！

恢复的 Stage 1 嵌入 (以 S 蛋白为例):
  - 形状: torch.Size([1, 1152])
  - 部分值: [ 0.19430137 -5.428516   -0.83095837 -7.2165313   1.2029688  -0.48015606
  0.7812158   3.8812513   0.95200455  2.7413125 ]

-- 最终生成的蛋白质序列 --

>NSP1 (长度: 180)
MANAVPGNNENTRLQLSLEVLQVRGVLKRGWDDSVEEKLSSARQHLKDGTCGLVEVETGVLNNLEQPYVFIKRSDARTAPHGHVTVEDLADLEGTQYGDSGETWS

In [7]:
random_stage2_embedding

tensor([ 0.3538,  0.2171,  1.0619,  ...,  3.1022, -0.3121, -0.9399])